In [1]:
# Imports
import joblib
import numpy as np 
import pandas as pd 
import sklearn
import xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from joblib import dump, load
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Carrega os dados
df = pd.read_csv('transaction_dataset.csv')

In [3]:
# Shape
df.shape

(9841, 51)

In [4]:
# Visualiza as primeiras linhas
df.head()

,Unnamed: 0,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


In [5]:
# Variável alvo
df.FLAG.value_counts()

FLAG
0    7662
1    2179
Name: count, dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 51 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Unnamed: 0                                            9841 non-null   int64  
 1   Index                                                 9841 non-null   int64  
 2   Address                                               9841 non-null   object 
 3   FLAG                                                  9841 non-null   int64  
 4   Avg min between sent tnx                              9841 non-null   float64
 5   Avg min between received tnx                          9841 non-null   float64
 6   Time Diff between first and last (Mins)               9841 non-null   float64
 7   Sent tnx                                              9841 non-null   int64  
 8   Received Tnx                                          9841

In [7]:
# Visualiza as primeiras linhas
df.head()

,Unnamed: 0,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


In [8]:
df.columns

Index(['Unnamed: 0', 'Index', 'Address', 'FLAG', 'Avg min between sent tnx',
       'Avg min between received tnx',
       'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx',
       'Number of Created Contracts', 'Unique Received From Addresses',
       'Unique Sent To Addresses', 'min value received', 'max value received ',
       'avg val received', 'min val sent', 'max val sent', 'avg val sent',
       'min value sent to contract', 'max val sent to contract',
       'avg value sent to contract',
       'total transactions (including tnx to create contract',
       'total Ether sent', 'total ether received',
       'total ether sent contracts', 'total ether balance',
       ' Total ERC20 tnxs', ' ERC20 total Ether received',
       ' ERC20 total ether sent', ' ERC20 total Ether sent contract',
       ' ERC20 uniq sent addr', ' ERC20 uniq rec addr',
       ' ERC20 uniq sent addr.1', ' ERC20 uniq rec contract addr',
       ' ERC20 avg time between sent tnx', ' ERC20 

In [9]:
# Ajusta o nome para minúsculo
df.columns = [x.lower() for x in df.columns]

In [10]:
df.columns

Index(['unnamed: 0', 'index', 'address', 'flag', 'avg min between sent tnx',
       'avg min between received tnx',
       'time diff between first and last (mins)', 'sent tnx', 'received tnx',
       'number of created contracts', 'unique received from addresses',
       'unique sent to addresses', 'min value received', 'max value received ',
       'avg val received', 'min val sent', 'max val sent', 'avg val sent',
       'min value sent to contract', 'max val sent to contract',
       'avg value sent to contract',
       'total transactions (including tnx to create contract',
       'total ether sent', 'total ether received',
       'total ether sent contracts', 'total ether balance',
       ' total erc20 tnxs', ' erc20 total ether received',
       ' erc20 total ether sent', ' erc20 total ether sent contract',
       ' erc20 uniq sent addr', ' erc20 uniq rec addr',
       ' erc20 uniq sent addr.1', ' erc20 uniq rec contract addr',
       ' erc20 avg time between sent tnx', ' erc20 

In [11]:
cols_to_drop = [' erc20 most sent token type',
                ' erc20_most_rec_token_type',
                'address',
                'index',
                'unnamed: 0']

In [12]:
# Seleciona os atributos filtrando as colunas que serão removidas e a variável alvo
atributos = [x for x in df.columns if (x != 'flag' and x not in cols_to_drop)]

In [13]:
atributos

['avg min between sent tnx',
 'avg min between received tnx',
 'time diff between first and last (mins)',
 'sent tnx',
 'received tnx',
 'number of created contracts',
 'unique received from addresses',
 'unique sent to addresses',
 'min value received',
 'max value received ',
 'avg val received',
 'min val sent',
 'max val sent',
 'avg val sent',
 'min value sent to contract',
 'max val sent to contract',
 'avg value sent to contract',
 'total transactions (including tnx to create contract',
 'total ether sent',
 'total ether received',
 'total ether sent contracts',
 'total ether balance',
 ' total erc20 tnxs',
 ' erc20 total ether received',
 ' erc20 total ether sent',
 ' erc20 total ether sent contract',
 ' erc20 uniq sent addr',
 ' erc20 uniq rec addr',
 ' erc20 uniq sent addr.1',
 ' erc20 uniq rec contract addr',
 ' erc20 avg time between sent tnx',
 ' erc20 avg time between rec tnx',
 ' erc20 avg time between rec 2 tnx',
 ' erc20 avg time between contract tnx',
 ' erc20 min val

In [14]:
# Extrai valores únicos
valores_unicos = df.nunique()

In [15]:
valores_unicos

unnamed: 0                                              9841
index                                                   4729
address                                                 9816
flag                                                       2
avg min between sent tnx                                5013
avg min between received tnx                            6223
time diff between first and last (mins)                 7810
sent tnx                                                 641
received tnx                                             727
number of created contracts                               20
unique received from addresses                           256
unique sent to addresses                                 258
min value received                                      4589
max value received                                      6302
avg val received                                        6767
min val sent                                            4719
max val sent            

In [16]:
# Mantém somente atributos com mais de um valor único (atributos que não são constantes)
atributos = [x for x in atributos if x in valores_unicos.loc[(valores_unicos > 1)]]

In [17]:
df[atributos].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 38 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   avg min between sent tnx                              9841 non-null   float64
 1   avg min between received tnx                          9841 non-null   float64
 2   time diff between first and last (mins)               9841 non-null   float64
 3   sent tnx                                              9841 non-null   int64  
 4   received tnx                                          9841 non-null   int64  
 5   number of created contracts                           9841 non-null   int64  
 6   unique received from addresses                        9841 non-null   int64  
 7   unique sent to addresses                              9841 non-null   int64  
 8   min value received                                    9841

In [18]:
# Definição de uma classe personalizada que herda de BaseEstimator e TransformerMixin
class PipeSteps(BaseEstimator, TransformerMixin):

    # Método construtor para inicializar a classe com uma lista de colunas
    def __init__(self, columns=[]):
        
        # Atribuição do argumento columns ao atributo de instância self.columns
        self.columns = columns

    # Método fit usado para ajustar (treinar) a transformação nos dados de treinamento
    def fit(self, X, y = None):
        
        # Retorna a própria instância, indicando que o método não faz modificações
        return self

    # Método transform para transformar os dados de entrada
    def transform(self, X):
        
        # Faz uma cópia dos dados de entrada para evitar alterar os dados originais
        X = X.copy()
        
        # Retorna os dados sem modificações
        return X

In [19]:
# Definição de uma classe que herda de PipeSteps
class SelecionaColunas(PipeSteps):

    # Método transform para transformar os dados de entrada
    def transform(self, X):
        
        # Faz uma cópia dos dados de entrada para evitar alterar os dados originais
        X = X.copy()
        
        # Seleciona e retorna apenas as colunas especificadas em self.columns
        return X[self.columns]

In [20]:
# Definição de uma classe que herda de PipeSteps
class PreencheDados(PipeSteps):

    # Método fit para ajustar a transformação nos dados de treinamento
    def fit(self, X, y = None):
        
        # Calcula a média de cada coluna especificada em self.columns e armazena no dicionário self.means
        self.means = { col: X[col].mean() for col in self.columns }
        
        # Retorna a própria instância, indicando que o método não faz modificações
        return self

    # Método transform para transformar os dados de entrada
    def transform(self, X):
        
        # Faz uma cópia dos dados de entrada para evitar alterar os dados originais
        X = X.copy()
        
        # Itera sobre cada coluna especificada em self.columns
        for col in self.columns:
            
            # Preenche valores ausentes na coluna com a média calculada na fase de ajuste
            X[col] = X[col].fillna(self.means[col])
        
        # Retorna os dados transformados
        return X

In [21]:
# Definição de uma classe que herda de PipeSteps
class PadronizaDados(PipeSteps):

    # Método fit para ajustar o scaler nos dados de treinamento
    def fit(self, X, y = None):
        
        # Inicializa uma instância de StandardScaler para padronizar os dados
        self.scaler = StandardScaler()
        
        # Ajusta o scaler nas colunas especificadas em self.columns
        self.scaler.fit(X[self.columns])
        
        # Retorna a própria instância, indicando que o método não faz modificações
        return self

    # Método transform para transformar os dados de entrada
    def transform(self, X):
        
        # Faz uma cópia dos dados de entrada para evitar alterar os dados originais
        X = X.copy()
        
        # Aplica a transformação de padronização nas colunas especificadas
        X[self.columns] = self.scaler.transform(X[self.columns])
        
        # Retorna os dados transformados
        return X

In [22]:
# Cria o pipeline de pré-processamento
_pipe_prepropcessamento = Pipeline([('feature_selection', SelecionaColunas(atributos)),
                                       ('fill_missing', PreencheDados(atributos)),
                                       ('standard_scaling', PadronizaDados(atributos))])

In [23]:
# Cria o pipeline de Machine Learning
pipe_final = Pipeline([
    ('preprocessing', _pipe_prepropcessamento),
    ('learning', XGBClassifier(random_state = 42, eval_metric = 'auc', objective = 'binary:logistic') )
])

In [24]:
# Variáveis de entrada
X = df[atributos]

In [25]:
# Variável de saída
y = df['flag']

In [26]:
# Divide os dados em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size = 0.30, random_state = 42)

In [27]:
X_treino

,avg min between sent tnx,avg min between received tnx,time diff between first and last (mins),sent tnx,received tnx,number of created contracts,unique received from addresses,unique sent to addresses,min value received,max value received,...,erc20 uniq sent addr.1,erc20 uniq rec contract addr,erc20 min val rec,erc20 max val rec,erc20 avg val rec,erc20 min val sent,erc20 max val sent,erc20 avg val sent,erc20 uniq sent token name,erc20 uniq rec token name
8460,0.00,0.00,0.00,0,0,0,0,0,0.000000,0.000000,...,0.0,1.0,1.337000,1.337000,1.337000,0.0,0.0,0.0,0.0,1.0
6081,0.00,8337.42,200098.17,0,24,1,3,0,0.000000,1.003651,...,0.0,2.0,0.301638,0.953298,0.627468,0.0,0.0,0.0,0.0,2.0
8966,0.00,318.17,2287.65,1,2,0,2,1,0.005000,0.300000,...,0.0,1.0,20000.000000,542000.000000,281000.000000,0.0,0.0,0.0,0.0,1.0
1535,28.33,3901.86,1021678.35,254,260,0,25,2,0.000047,110.736000,...,0.0,5.0,0.000000,19421.000000,3961.659270,0.0,0.0,0.0,0.0,5.0
7304,0.00,1947.10,319323.80,0,164,1,4,0,0.000000,1.903509,...,0.0,3.0,0.000000,2.335693,1.313279,0.0,0.0,0.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,0.00,16197.17,226760.43,0,14,1,6,0,0.000000,14.419115,...,0.0,2.0,0.518689,0.705159,0.611924,0.0,0.0,0.0,0.0,2.0
5191,0.00,0.00,15369.12,1,1,0,1,1,2.000000,2.000000,...,0.0,7.0,0.000000,312.430205,50.497598,0.0,0.0,0.0,0.0,7.0
5390,0.00,0.00,1.77,1,1,0,1,1,1.990000,1.990000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
860,165.15,0.00,330.30,2,2,0,2,2,49.770407,51.229593,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


In [28]:
# Treino do modelo
pipe_final.fit(X_treino, y_treino)

,steps,"[('preprocessing', ...), ('learning', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('feature_selection', ...), ('fill_missing', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,columns,"['avg min between sent tnx', 'avg min between received tnx', ...]"
,columns,"['avg min between sent tnx', 'avg min between received tnx', ...]"
,columns,"['avg min between sent tnx', 'avg min between received tnx', ...]"


In [29]:
# Previsões com dados de teste
previsoes_teste = pipe_final.predict(X_teste)

In [30]:
previsoes_teste

array([1, 1, 0, ..., 0, 0, 0])

In [31]:
# Calcula a métrica AUC
score_auc = metrics.roc_auc_score(y_teste, previsoes_teste)

In [32]:
print(f'AUC nos Dados de Teste - {score_auc:,.2%}')

AUC nos Dados de Teste - 97.21%


In [33]:
# Salva o modelo em disco
dump(pipe_final, 'modelo_final.joblib')

['modelo_final.joblib']

In [34]:
# Carrega os novos dados de uma transação
novos_dados = pd.read_csv('dados_teste.csv')

In [35]:
# Dados no formato original
novos_dados

,avg min between sent tnx,avg min between received tnx,time diff between first and last (mins),sent tnx,received tnx,number of created contracts,unique received from addresses,unique sent to addresses,min value received,max value received,...,erc20 uniq sent addr.1,erc20 uniq rec contract addr,erc20 min val rec,erc20 max val rec,erc20 avg val rec,erc20 min val sent,erc20 max val sent,erc20 avg val sent,erc20 uniq sent token name,erc20 uniq rec token name
0,2570.59,3336.01,30572.7,8,3,0,2,4,0.1,40.0,...,0.0,1.0,600.0,600.0,600.0,0.0,0.0,0.0,0.0,1.0


In [36]:
# Carrega o modelo do disco
modelo_carregado = load('modelo_final.joblib')

In [37]:
# Extrai a previsão de maior probabilidade
previsao = modelo_carregado.predict(novos_dados)

In [38]:
# Resultado
if previsao[0] == 0:
    print("Segundo o modelo, provavelmente, essa transação não representa uma Fraude.")
else:
    print("Segundo o modelo, provavelmente, essa transação pode representar uma Fraude. Acione verificação humana!")

Segundo o modelo, provavelmente, essa transação não representa uma Fraude.
